In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import quad
import bilby
import corner
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

np.random.seed(21)

## Cosmology Constants

In [2]:
H0_KM_S_MPC = 67.36
OMEGA_M = 0.3153
OMEGA_L = 0.6847
MPC_IN_KM = 3.0856775814913673e19          # 1 Mpc in km
H0_INV_S = MPC_IN_KM / H0_KM_S_MPC          # 1/H0 in seconds

## Custom Likelihood for the Time-Delay Model


In [3]:
class GRBTimeDelayLikelihood(bilby.Likelihood):
    def __init__(self, dt_obs, sigma_dt_obs, E_s, sigma_E_s, K1, sigma_K1):
        
        super().__init__(parameters={'a_LV': None, 'alpha': None, 'mu': None, 'v': None})
        self.dt_obs = dt_obs
        self.sigma_dt_obs = sigma_dt_obs
        self.E_s = E_s
        self.sigma_E_s = sigma_E_s
        self.K1 = K1
        self.sigma_K1 = sigma_K1

    def log_likelihood(self, parameters=None):
        
        if parameters is not None:
            self.parameters.update(parameters)
            
        a_LV = self.parameters['a_LV']
        alpha = self.parameters['alpha']
        mu = self.parameters['mu']
        v = self.parameters['v']
        
        # Mean model expectation
        model_dt = a_LV * self.K1 + alpha * self.E_s + mu
        
        # Propagated variance
        variance = (a_LV**2) * (self.sigma_K1**2) + (alpha**2) * (self.sigma_E_s**2) + (v**2) + (self.sigma_dt_obs**2)
        
        # Gaussian log-likelihood
        log_l = -0.5 * np.sum(((self.dt_obs - model_dt)**2) / variance + np.log(2 * np.pi * variance))
        return log_l

## Input Data 

In [4]:
# # 14 Multi-GeV Photons (Label I)
# dt_obs_14 = np.array([1.98, 6.45, 0.45, 25.5, 0.67, 1.56, 5.81, 11.55, 12.70, 6.60, 13.97, 13.51, 0.14, 28.84])
# sigma_dt_14 = np.array([0.93, 0.93, 2.63, 1.77, 1.77, 1.77, 1.77, 1.77, 1.77, 1.61, 2.11, 3.73, 1.36, 2.30])

# E_s_14 = np.array([66.34, 146.60, 56.90, 112.60, 33.58, 40.07, 51.08, 35.84, 43.46, 60.59, 70.33, 97.28, 83.31, 112.62])
# sigma_E_14 = np.array([6.63, 14.66, 5.69, 11.26, 3.36, 4.01, 5.11, 3.58, 4.35, 6.06, 7.03, 9.73, 8.33, 11.26])

# K1_14 = np.array([4.45, 9.83, 7.19, 12.84, 3.83, 4.57, 5.83, 4.09, 4.96, 6.51, 8.68, 8.99, 7.92, 14.20])
# sigma_K1_14 = np.array([0.45, 0.99, 0.72, 1.29, 0.39, 0.46, 0.59, 0.41, 0.50, 0.66, 0.87, 0.90, 0.80, 1.43])

# # 3 Remarkable Photons (Label II)
# dt_obs_3 = np.array([-8.71, 48.86, 340.19])
# sigma_dt_3 = np.array([4.35, 3.51, 4.35])

# E_s_3 = np.array([114.29, 1524.21, 14042.20])
# sigma_E_3 = np.array([11.43, 228.63, 2808.44])

# K1_3 = np.array([6.15, 157.30, 755.66])
# sigma_K1_3 = np.array([0.62, 23.65, 151.31])


In [5]:
# Load the dataset
df = pd.read_csv('data.csv')
print(list(df.columns))

['Label', 'Observatory', 'GRB', 'z', 'E_high_o', 'E_high_o_err', 'T_high_o', 'T_low_o', 'dT_low_err', 'dt_obs', 'dt_obs_err', 'E_high_s', 'E_high_s_err', 'K1', 'K1_err']


In [6]:
# 14 Multi-GeV Photons (Label I)
df_14 = df[df['Label'] == 'I']

dt_obs_14   = df_14['dt_obs'].to_numpy()
sigma_dt_14 = df_14['dt_obs_err'].to_numpy()
E_s_14      = df_14['E_high_s'].to_numpy()
sigma_E_14  = df_14['E_high_s_err'].to_numpy()
K1_14       = df_14['K1'].to_numpy()
sigma_K1_14 = df_14['K1_err'].to_numpy()

# 3 Remarkable Photons (Label II)
df_3 = df[df['Label'] == 'II']

dt_obs_3   = df_3['dt_obs'].to_numpy()
sigma_dt_3 = df_3['dt_obs_err'].to_numpy()
E_s_3      = df_3['E_high_s'].to_numpy()
sigma_E_3  = df_3['E_high_s_err'].to_numpy()
K1_3       = df_3['K1'].to_numpy()
sigma_K1_3 = df_3['K1_err'].to_numpy()

## Carpet-3 300 TeV Photon 

In [7]:
def _cosmo_integrand(zp):
    return (1.0 + zp) / np.sqrt(OMEGA_M * (1.0 + zp) ** 3 + OMEGA_L)


def compute_K1(z, E_high_o_GeV):
    integral, _ = quad(_cosmo_integrand, 0.0, z)
    K1_s_GeV = H0_INV_S * E_high_o_GeV / (1.0 + z) * integral
    return K1_s_GeV / 1e18

In [8]:
z_carpet = 0.151
E_high_o_carpet = 300.0e3       # GeV (300 TeV)
E_high_o_err_carpet = 0.13 * E_high_o_carpet   # 13% energy resolution (in the paper)
T_high_o_carpet = 4536.0
T_low_o_carpet = 251.33
dT_low_err_carpet = 5.00

dt_obs_carpet = np.array([(T_high_o_carpet - T_low_o_carpet) / (1.0 + z_carpet)])
sigma_dt_carpet = np.array([dT_low_err_carpet / (1.0 + z_carpet)])

E_s_carpet = np.array([E_high_o_carpet * (1.0 + z_carpet)])
sigma_E_carpet = np.array([E_high_o_err_carpet * (1.0 + z_carpet)]) 

K1_carpet = np.array([compute_K1(z_carpet, E_high_o_carpet)]) 
sigma_K1_carpet = K1_carpet * (E_high_o_err_carpet / E_high_o_carpet)

## Combine Data Scenarios

In [9]:

# Case 1a: 14 Photons (Figure 1a)
dt_obs_fig1a = dt_obs_14
sigma_dt_fig1a = sigma_dt_14
E_s_fig1a = E_s_14
sigma_E_fig1a = sigma_E_14
K1_fig1a = K1_14
sigma_K1_fig1a = sigma_K1_14

# Case 1b: 14 Photons + Carpet-3 (Figure 1b)
dt_obs_fig1b = np.concatenate((dt_obs_14, dt_obs_carpet))
sigma_dt_fig1b = np.concatenate((sigma_dt_14, sigma_dt_carpet))
E_s_fig1b = np.concatenate((E_s_14, E_s_carpet))
sigma_E_fig1b = np.concatenate((sigma_E_14, sigma_E_carpet))
K1_fig1b = np.concatenate((K1_14, K1_carpet))
sigma_K1_fig1b = np.concatenate((sigma_K1_14, sigma_K1_carpet))

# case 2a: 17 Photons (Figure 2a)
dt_obs_fig2a = np.concatenate((dt_obs_14, dt_obs_3))
sigma_dt_fig2a = np.concatenate((sigma_dt_14, sigma_dt_3))
E_s_fig2a = np.concatenate((E_s_14, E_s_3))
sigma_E_fig2a = np.concatenate((sigma_E_14, sigma_E_3))
K1_fig2a = np.concatenate((K1_14, K1_3))
sigma_K1_fig2a = np.concatenate((sigma_K1_14, sigma_K1_3))

# Case 2b: 17 Photons + Carpet-3 (Figure 2b)
dt_obs_fig2b = np.concatenate((dt_obs_14, dt_obs_3, dt_obs_carpet))
sigma_dt_fig2b = np.concatenate((sigma_dt_14, sigma_dt_3, sigma_dt_carpet))
E_s_fig2b = np.concatenate((E_s_14, E_s_3, E_s_carpet))
sigma_E_fig2b = np.concatenate((sigma_E_14, sigma_E_3, sigma_E_carpet))
K1_fig2b = np.concatenate((K1_14, K1_3, K1_carpet))
sigma_K1_fig2b = np.concatenate((sigma_K1_14, sigma_K1_3, sigma_K1_carpet))

## Uniform Priors

In [10]:
priors = bilby.core.prior.PriorDict()
priors['a_LV'] = bilby.core.prior.Uniform(minimum=0, maximum=30, name='a_LV')
priors['alpha'] = bilby.core.prior.Uniform(minimum=-30, maximum=30, name='alpha')
priors['mu'] = bilby.core.prior.Uniform(minimum=-30, maximum=30, name='mu')
priors['v'] = bilby.core.prior.Uniform(minimum=0, maximum=30, name='v')

## MCMC Sampling

In [11]:
likelihood_fig1a = GRBTimeDelayLikelihood(dt_obs_fig1a, sigma_dt_fig1a, E_s_fig1a, sigma_E_fig1a, K1_fig1a, sigma_K1_fig1a)

result_fig1a = bilby.run_sampler(
    likelihood=likelihood_fig1a, priors=priors, sampler='dynesty',
    nlive=3000, outdir='outdir', label='fig1a_reproduction'
)

result_fig1a.plot_corner(
    labels=[r"$a_{LV} \; [10^{-18} GeV^{-1}]$", r"$\alpha \; [s \cdot GeV^{-1}]$", r"$\mu \; [s]$", r"$v \; [s]$"],
    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 12}
)
plt.show()


18:43 bilby INFO    : Running for label 'fig1a_reproduction', output will be saved to 'outdir'
18:43 bilby INFO    : Analysis priors:
18:43 bilby INFO    : a_LV=Uniform(minimum=0, maximum=30, name='a_LV', latex_label='a_LV', unit=None, boundary=None)
18:43 bilby INFO    : alpha=Uniform(minimum=-30, maximum=30, name='alpha', latex_label='alpha', unit=None, boundary=None)
18:43 bilby INFO    : mu=Uniform(minimum=-30, maximum=30, name='mu', latex_label='mu', unit=None, boundary=None)
18:43 bilby INFO    : v=Uniform(minimum=0, maximum=30, name='v', latex_label='v', unit=None, boundary=None)
18:43 bilby INFO    : Analysis likelihood class: <class '__main__.GRBTimeDelayLikelihood'>
18:43 bilby INFO    : Analysis likelihood noise evidence: nan
18:43 bilby INFO    : Single likelihood evaluation took 2.221e-04 s
18:43 bilby INFO    : Using sampler Dynesty with kwargs {'nlive': 3000, 'bound': 'live', 'sample': 'act-walk', 'periodic': None, 'reflective': None, 'update_interval': 600, 'first_updat

1it [00:00, ?it/s]

18:50 bilby INFO    : Written checkpoint file outdir/fig1a_reproduction_resume.pickle
18:51 bilby INFO    : Rejection sampling nested samples to obtain 13094 posterior samples
18:51 bilby INFO    : Sampling time: 0:07:55.765398
18:51 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 


18:51 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
18:51 bilby INFO    : Summary of results:
nsamples: 13094
ln_noise_evidence:    nan
ln_evidence: -56.634 +/-  0.081
ln_bayes_factor:    nan +/-  0.081



In [12]:
likelihood_fig1b = GRBTimeDelayLikelihood(dt_obs_fig1b, sigma_dt_fig1b, E_s_fig1b, sigma_E_fig1b, K1_fig1b, sigma_K1_fig1b)

result_fig1b = bilby.run_sampler(
    likelihood=likelihood_fig1b, priors=priors, sampler='dynesty',
    nlive=3000, outdir='outdir', label='fig1b_reproduction'
)

result_fig1b.plot_corner(
    labels=[r"$a_{LV} \; [10^{-18} GeV^{-1}]$", r"$\alpha \; [s \cdot GeV^{-1}]$", r"$\mu \; [s]$", r"$v \; [s]$"],
    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 12}
)
plt.show()

18:51 bilby INFO    : Running for label 'fig1b_reproduction', output will be saved to 'outdir'
18:51 bilby INFO    : Analysis priors:
18:51 bilby INFO    : a_LV=Uniform(minimum=0, maximum=30, name='a_LV', latex_label='a_LV', unit=None, boundary=None)
18:51 bilby INFO    : alpha=Uniform(minimum=-30, maximum=30, name='alpha', latex_label='alpha', unit=None, boundary=None)
18:51 bilby INFO    : mu=Uniform(minimum=-30, maximum=30, name='mu', latex_label='mu', unit=None, boundary=None)
18:51 bilby INFO    : v=Uniform(minimum=0, maximum=30, name='v', latex_label='v', unit=None, boundary=None)
18:51 bilby INFO    : Analysis likelihood class: <class '__main__.GRBTimeDelayLikelihood'>
18:51 bilby INFO    : Analysis likelihood noise evidence: nan
18:51 bilby INFO    : Single likelihood evaluation took 2.808e-04 s
18:51 bilby INFO    : Using sampler Dynesty with kwargs {'nlive': 3000, 'bound': 'live', 'sample': 'act-walk', 'periodic': None, 'reflective': None, 'update_interval': 600, 'first_updat

1it [00:00, ?it/s]

18:55 bilby INFO    : Written checkpoint file outdir/fig1b_reproduction_resume.pickle
18:55 bilby INFO    : Rejection sampling nested samples to obtain 12830 posterior samples
18:55 bilby INFO    : Sampling time: 0:04:21.969618
18:55 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 


18:55 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
18:55 bilby INFO    : Summary of results:
nsamples: 12830
ln_noise_evidence:    nan
ln_evidence: -67.572 +/-  0.083
ln_bayes_factor:    nan +/-  0.083



In [13]:
likelihood_fig2a = GRBTimeDelayLikelihood(dt_obs_fig2a, sigma_dt_fig2a, E_s_fig2a, sigma_E_fig2a, K1_fig2a, sigma_K1_fig2a)

result_fig2a = bilby.run_sampler(
    likelihood=likelihood_fig2a, priors=priors, sampler='dynesty',
    nlive=3000, outdir='outdir', label='fig2a_reproduction'
)

result_fig2a.plot_corner(
    labels=[r"$a_{LV} \; [10^{-18} GeV^{-1}]$", r"$\alpha \; [s \cdot GeV^{-1}]$", r"$\mu \; [s]$", r"$v \; [s]$"],
    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 12}
)
plt.show()

18:55 bilby INFO    : Running for label 'fig2a_reproduction', output will be saved to 'outdir'
18:55 bilby INFO    : Analysis priors:
18:55 bilby INFO    : a_LV=Uniform(minimum=0, maximum=30, name='a_LV', latex_label='a_LV', unit=None, boundary=None)
18:55 bilby INFO    : alpha=Uniform(minimum=-30, maximum=30, name='alpha', latex_label='alpha', unit=None, boundary=None)
18:55 bilby INFO    : mu=Uniform(minimum=-30, maximum=30, name='mu', latex_label='mu', unit=None, boundary=None)
18:55 bilby INFO    : v=Uniform(minimum=0, maximum=30, name='v', latex_label='v', unit=None, boundary=None)
18:55 bilby INFO    : Analysis likelihood class: <class '__main__.GRBTimeDelayLikelihood'>
18:55 bilby INFO    : Analysis likelihood noise evidence: nan
18:55 bilby INFO    : Single likelihood evaluation took 2.498e-04 s
18:55 bilby INFO    : Using sampler Dynesty with kwargs {'nlive': 3000, 'bound': 'live', 'sample': 'act-walk', 'periodic': None, 'reflective': None, 'update_interval': 600, 'first_updat

1it [00:00, ?it/s]

19:00 bilby INFO    : Written checkpoint file outdir/fig2a_reproduction_resume.pickle


19:00 bilby INFO    : Rejection sampling nested samples to obtain 12291 posterior samples
19:00 bilby INFO    : Sampling time: 0:04:55.268559
19:00 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
19:00 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
19:00 bilby INFO    : Summary of results:
nsamples: 12291
ln_noise_evidence:    nan
ln_evidence: -75.880 +/-  0.083
ln_bayes_factor:    nan +/-  0.083



In [14]:
likelihood_fig2b = GRBTimeDelayLikelihood(dt_obs_fig2b, sigma_dt_fig2b, E_s_fig2b, sigma_E_fig2b, K1_fig2b, sigma_K1_fig2b)

result_fig2b = bilby.run_sampler(
    likelihood=likelihood_fig2b, priors=priors, sampler='dynesty',
    nlive=3000, outdir='outdir', label='fig2b_reproduction'
)

result_fig2b.plot_corner(
    labels=[r"$a_{LV} \; [10^{-18} GeV^{-1}]$", r"$\alpha \; [s \cdot GeV^{-1}]$", r"$\mu \; [s]$", r"$v \; [s]$"],
    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_kwargs={"fontsize": 12}
)
plt.show()

19:00 bilby INFO    : Running for label 'fig2b_reproduction', output will be saved to 'outdir'
19:00 bilby INFO    : Analysis priors:
19:00 bilby INFO    : a_LV=Uniform(minimum=0, maximum=30, name='a_LV', latex_label='a_LV', unit=None, boundary=None)
19:00 bilby INFO    : alpha=Uniform(minimum=-30, maximum=30, name='alpha', latex_label='alpha', unit=None, boundary=None)
19:00 bilby INFO    : mu=Uniform(minimum=-30, maximum=30, name='mu', latex_label='mu', unit=None, boundary=None)
19:00 bilby INFO    : v=Uniform(minimum=0, maximum=30, name='v', latex_label='v', unit=None, boundary=None)
19:00 bilby INFO    : Analysis likelihood class: <class '__main__.GRBTimeDelayLikelihood'>
19:00 bilby INFO    : Analysis likelihood noise evidence: nan
19:00 bilby INFO    : Single likelihood evaluation took 6.694e-04 s
19:00 bilby INFO    : Using sampler Dynesty with kwargs {'nlive': 3000, 'bound': 'live', 'sample': 'act-walk', 'periodic': None, 'reflective': None, 'update_interval': 600, 'first_updat

1it [00:00, ?it/s]

19:05 bilby INFO    : Written checkpoint file outdir/fig2b_reproduction_resume.pickle
19:06 bilby INFO    : Rejection sampling nested samples to obtain 12139 posterior samples


19:06 bilby INFO    : Sampling time: 0:05:23.072176
19:06 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
19:06 bilby WARNING : Result.save_to_file called with extension=True. This will default to json, and ignore the extension from the filename. This behaviour is deprecated and will be removed. 
19:06 bilby INFO    : Summary of results:
nsamples: 12139
ln_noise_evidence:    nan
ln_evidence: -86.621 +/-  0.084
ln_bayes_factor:    nan +/-  0.084



In [ ]:
# Dictionary of your 4 Bilby result objects
results = {
    'Fig 1a': result_fig1a,
    'Fig 1b': result_fig1b,
    'Fig 2a': result_fig2a,
    'Fig 2b': result_fig2b,
}

param_names = ['a_LV', 'alpha', 'mu', 'v']

summary_rows = []

for case_name, res in results.items():
    row = {'Case': case_name}
    for param in param_names:
        if param in res.posterior:
            # Get samples for the current parameter
            samples = res.posterior[param].values
            q16, q50, q84 = np.percentile(samples, [16, 50, 84])
            plus_err = q84 - q50
            minus_err = q50 - q16
            
            # Format as: median +plus -minus
            row[param] = f"{q50:.2f} +{plus_err:.2f} / -{minus_err:.2f}"
            
            if param == 'a_LV':
                # Convert a_LV samples to E_LV (in units of 10^17 GeV)
                e_lv_samples = 10.0 / samples
                
                # Calculate percentiles for E_LV
                eq16, eq50, eq84 = np.percentile(e_lv_samples, [16, 50, 84])
                e_plus_err = eq84 - eq50
                e_minus_err = eq50 - eq16
                
                row['E_LV'] = f"{eq50:.2f} +{e_plus_err:.2f} / -{e_minus_err:.2f}"
        else:
            row[param] = "N/A"
            
    summary_rows.append(row)

# Create DataFrame
summary_df = pd.DataFrame(summary_rows)

# Reorder columns so E_LV is right next to a_LV
cols = ['Case', 'a_LV', 'E_LV', 'alpha', 'mu', 'v']
summary_df = summary_df[cols]

# Print console table
from IPython.display import display

# Display nicely rendered HTML table
summary_df.style.hide(axis='index')

Case,a_LV,E_LV,alpha,mu,v
Fig 1a,3.28 +1.07 / -0.95,3.05 +1.25 / -0.75,-0.15 +0.09 / -0.11,-4.50 +4.23 / -4.43,5.27 +1.95 / -1.50
Fig 1b,3.32 +0.95 / -0.83,3.01 +1.00 / -0.67,-0.16 +0.05 / -0.07,-3.37 +3.49 / -3.60,5.05 +1.86 / -1.41
Fig 2a,3.33 +0.91 / -0.89,3.00 +1.10 / -0.64,-0.20 +0.06 / -0.07,-1.05 +3.51 / -3.53,5.08 +1.98 / -1.51
Fig 2b,3.03 +0.91 / -1.05,3.30 +1.74 / -0.76,-0.17 +0.07 / -0.06,-1.43 +3.55 / -3.33,5.26 +2.14 / -1.54
